In [ ]:
FILE_NAME='data/docx/sample-docx.docx'

Podejścia do odczytu pliku DOCX w Pythonie:
1. Konwersja do HTML za pomocą mammoth.js, a następnie parsowanie przy użyciu BeautifulSoup.
2. Konwersja do Markdown za pomocą MarkItDown, a następnie użycie parsera Markdown.
3. Użycie unstructured.io do bezpośredniego wyodrębniania i przetwarzania dokumentu.
4. Użycie bibliotek takich jak python-docx lub docling do odczytu struktury dokumentu.

## mammoth.js

**Mammoth.js** to biblioteka JavaScript zaprojektowana do konwersji dokumentów `.docx` do czystego, semantycznego HTML. Zamiast odtwarzać style wizualne, Mammoth skupia się na znaczącej strukturze - na przykład zamienia „Heading 1” na tagi `<h1>`. Obsługuje między innymi:

*   nagłówki, listy, tabele, przypisy, obrazy, linki
*   mapowanie niestandardowych stylów
*   wyodrębnianie surowego tekstu
*   użycie w przeglądarce i w Node.js
*   obsługę CLI i API

Mammoth unika zbędnego formatowania i działa najlepiej z dokumentami mającymi semantyczny układ stylów. Jest dostępny przez npm, PyPI, Maven, NuGet i WordPress.



In [ ]:
%pip install mammoth

In [ ]:
import mammoth
from pathlib import Path
from bs4 import BeautifulSoup
from textwrap import shorten

output_path = Path(FILE_NAME).with_suffix(".html")

with open(FILE_NAME, "rb") as docx_file:
    result = mammoth.convert_to_html(docx_file)

html = result.value
messages = result.messages

output_path.write_text(html, encoding="utf-8")
print(f"Zapisano HTML w pliku {output_path}")

if messages:
    print("Komunikaty konwersji:")
    for message in messages:
        print(f" - {message}")

soup = BeautifulSoup(html, "html.parser")

headings = [tag.get_text(' ', strip=True) for tag in soup.find_all(["h1", "h2", "h3"])]
if headings:
    print("\nWykryte nagłówki:")
    for heading in headings:
        print(f" - {heading}")

paragraphs = [p.get_text(' ', strip=True) for p in soup.find_all("p")]
if paragraphs:
    print(f"\nFragment pierwszego akapitu: {shorten(paragraphs[0], width=120, placeholder='…')}")

tables = soup.find_all("table")
if tables:
    first_table = tables[0]
    rows = [
        [cell.get_text(' ', strip=True) for cell in row.find_all(["th", "td"])]
        for row in first_table.find_all("tr")
    ]
    print("\nPodgląd pierwszej tabeli:")
    for row in rows[:3]:
        print(" | ".join(row))

## MarkItDown 

**MarkItDown** to lekki, napisany w Pythonie zestaw narzędzi opracowany przez Microsoft do konwersji różnych formatów plików do Markdown, zoptymalizowany pod kątem pracy z dużymi modelami językowymi (LLM) i potokami analizy tekstu. Zachowuje strukturę dokumentu (nagłówki, tabele, listy itp.) i obsługuje szeroką gamę formatów wejściowych, w tym:

*   PDF, Word, Excel, PowerPoint
*   obrazy (z OCR i metadanymi EXIF)
*   audio (transkrypcja)
*   HTML, CSV, JSON, XML
*   pliki ZIP, adresy URL YouTube, EPUB-y

#### **Najważniejsze funkcje**

*   konwersja plików do Markdown dla wydajnego przetwarzania przez LLM
*   obsługa wtyczek i integracji z Azure Document Intelligence
*   dostępność zarówno w CLI, jak i w postaci API Pythona
*   zgodność z Dockerem i środowiskami wirtualnymi
*   opcjonalne zależności dla obsługi konkretnych formatów
*   obsługa opisów obrazów generowanych przez LLM (np. GPT-4o)
*   projekt open source na licencji MIT

In [ ]:
%pip install markitdown[all]

In [ ]:
from pathlib import Path
from markitdown import MarkItDown

converter = MarkItDown()  # Ustaw enable_plugins=True, jeśli potrzebujesz zaawansowanych formatów
result = converter.convert(FILE_NAME)

output_path = Path(FILE_NAME).with_suffix(".md")
output_path.write_text(result.text_content, encoding="utf-8")
print(f"Zapisano Markdown w pliku {output_path}")

lines = result.text_content.splitlines()
preview_lines = lines[:15] if lines else []
if preview_lines:
    print("\nPodgląd Markdown:")
    for line in preview_lines:
        print(line)
else:
    print("\nNie wykryto treści tekstowej.")

attachments = getattr(result, "attachments", None)
if attachments:
    print("\nOsadzone załączniki:")
    for name, data in attachments.items():
        print(f" - {name} ({len(data)} bajtów)")
else:
    print("\nNie wykryto osadzonych załączników.")

## unstructured.io

**Unstructured.io** to otwartoźródłowa biblioteka ETL zaprojektowana do konwersji złożonych dokumentów - takich jak PDF-y, pliki Word, HTML i obrazy - do czystych, ustrukturyzowanych danych zoptymalizowanych pod kątem użycia z dużymi modelami językowymi (LLM). Udostępnia modułowe komponenty do:

*   **wczytywania dokumentów i wstępnego przetwarzania**
*   **automatycznego dzielenia na partie i wykrywania formatu**
*   **wzbogacania tabel i obrazów**
*   **dzielenia na fragmenty i generowania embeddingów**

Unstructured obsługuje zarówno lokalne środowiska deweloperskie, jak i wdrożenia kontenerowe przez Docker. Łatwo integruje się z Pythonem i oferuje konektory dla platform takich jak Discord. Biblioteka jest dobrze dopasowana do budowania skalowalnych, produkcyjnych potoków danych i jest dostępna przez PyPI oraz GitHub.

In [ ]:
%pip  install "unstructured[docx]"

In [ ]:
from unstructured.partition.docx import partition_docx
elements = partition_docx(FILE_NAME)
print(elements)

In [ ]:
print("Liczba elementów: ", len(elements))
for i, element in enumerate(elements):             
    if element.category == 'Table':
        chunk_text = element.metadata.text_as_html        
    else:
        if element.category == 'Title':
            chunk_text = "# "+ element.text
        else:
            chunk_text = element.text 
    print(f'element {i} ({element.category}): długość fragmentu ({len(chunk_text)}) {chunk_text[:100]}...') 
    

### 🆚 Porównanie narzędzi: MarkItDown vs Unstructured.io vs Mammoth.js

| Funkcja / aspekt         | **MarkItDown**                                         | **Unstructured.io**                                        | **Mammoth.js**                                        |
| ------------------------ | ------------------------------------------------------ | ---------------------------------------------------------- | ----------------------------------------------------- |
| **Cel**                 | Konwertuje pliki do Markdown jako wejście przyjazne dla LLM | Przekształca nieustrukturyzowane dokumenty w ustrukturyzowane dane | Konwertuje pliki `.docx` do czystego HTML-a |
| **Obsługiwane formaty** | PDF, Word, Excel, PPT, obrazy, audio, HTML, JSON itd.  | PDF-y, Word, PowerPoint, HTML, e-maile, obrazy, EPUB-y itd. | Tylko `.docx` (Word) |
| **Format wyjściowy**    | Markdown (`.md`)                                       | Ustrukturyzowany JSON                                           | HTML (semantyczny, minimalny markup) |
| **Instalacja**          | `pip install 'markitdown[all]'`                        | `pip install "unstructured[all-docs]"`                     | `npm install mammoth` (także w Pythonie, Javie, .NET itp.) |
| **Sposób użycia**       | CLI i API Pythona                                     | SDK Pythona, CLI, Docker, API, interfejs webowy              | Node.js/JavaScript lub CLI; opcjonalny wrapper w Pythonie |
| **Nacisk przetwarzania** | Zachowuje strukturę dokumentu (nagłówki, tabele, listy) | Dzielenie, czyszczenie, chunking, ekstrakcja metadanych      | Skupienie na semantycznym mapowaniu stylów Word do HTML |
| **Obsługa obrazów i audio** | OCR dla obrazów, transkrypcja audio                | OCR, analiza układu, ekstrakcja tabel                      | Obrazy osadzone w wyjściowym HTML |
| **Możliwości dostosowania** | Wtyczki, integracja z Azure                           | Mapowanie stylów, konektory, orkiestracja potoków            | Własne mapy stylów do konwersji nazwanych stylów na HTML |
| **Funkcje enterprise**  | Lekkie narzędzie z obsługą wtyczek                     | Pełna platforma enterprise z UI, API, bezpieczeństwem, analityką | Rozwiązanie społecznościowe / open-source (wspierane przez GitHub) |
| **Open source**         | Tak (wspierany przez Microsoft)                        | Tak (rozwijany przez społeczność, hostowany na GitHubie)     | Tak (licencja BSD, wieloplatformowe) |
| **Wsparcie Dockera**    | Tak                                                    | Tak (obrazy wieloplatformowe)                               | Wsparcie dla demo w przeglądarce; podstawowe API działa w Node.js |
| **Najlepszy przypadek użycia** | Szybka konwersja do Markdown do przetwarzania przez LLM | Budowa pełnych potoków ETL dla AI/ML                         | Czysta konwersja Word `.docx` do semantycznego HTML   |
